<div align="center">

<a href="https://colab.research.google.com/github/utkukose/rd-project-management-NB-lecture/blob/main/weeks/week-09/NB09_cpm_pert.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 09: Scheduling: Critical Path and PERT

**R&D and Project Management in Computer Science (11117BLG002)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/rd-project-management-NB-lecture/weeks/week-09/lab.html>.

## Overview

Once tasks and dependencies are known, the schedule follows from arithmetic. This week introduces network diagrams, the critical path method and PERT, and discusses how schedules can be compressed and where their assumptions fail [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to draw a dependency network, to compute earliest and latest start and finish times and total float, to identify the critical path, to compute the probability of meeting a deadline with PERT, and to explain crashing, fast tracking and the limits of PERT.

## Networks of tasks

A schedule begins with dependencies. In an activity-on-node network, each task is a node and an arrow from one task to another means that the second depends on the first. The most common dependency is finish-to-start, in which a task can start only when its predecessor has finished; start-to-start, finish-to-finish and start-to-finish dependencies describe other relations. In research projects, dependencies often run through approvals and data: Experiments cannot start before data access is granted, and a user study cannot start before ethics approval.

## The critical path method

Kelley and Walker described the critical path method for planning and scheduling projects [1]. A forward pass through the network computes the earliest start and finish of every task: A task starts as soon as all its predecessors have finished. A backward pass from the project end computes the latest start and finish that do not delay the project. The difference between latest and earliest start is the total float, the delay a task can absorb without delaying the project. Tasks with zero float form the critical path, the longest path through the network, whose length is the project duration. Figure 9.1 shows the schedule of a small research project with two critical paths that merge before the experiments.

![Figure 9.1](https://raw.githubusercontent.com/utkukose/rd-project-management-NB-lecture/main/weeks/week-09/figures/w09_fig1.png)

*Figure 9.1. Critical path schedule of the example research project: two critical paths of 20 weeks merge at the prototype; benchmark design and workshop preparation have float [1].*

## PERT and the probability of meeting a deadline

PERT was developed for the research and development programme of the Polaris missile system [2]. It combines the network with three-point estimates: The expected duration of the project is the sum of the expected durations on the critical path, and its variance is the sum of their variances, if tasks are independent. With a normal approximation, the probability of finishing by a deadline follows. The method has a known weakness: It considers only one critical path, so it ignores the chance that a near-critical path becomes critical. When paths merge, the expected completion time is later than PERT suggests. Week 10 addresses this merge bias with Monte Carlo simulation.

## Compressing a schedule

When the computed duration exceeds the time available, the schedule can be compressed. Crashing adds resources to critical tasks to shorten them, at extra cost; fast tracking overlaps tasks that were planned in sequence, at extra risk [3]. Shortening a non-critical task gains nothing, and shortening a critical task can make another path critical. In research projects, the most effective compression is often to start long administrative tasks, such as data agreements and ethics applications, as early as possible.

> **Pause and reflect.** Which task of your project lies on the critical path, and what could you start earlier to protect it?

# Hands-on lab

The notebook implements the critical path method, computes the float of every task and estimates the probability of meeting a deadline with PERT [1, 2].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. The example network

In [ ]:
import pandas as pd
NET = [("A", "Literature review", 3, []), ("B", "Requirements and data access", 2, []), ("C", "Data collection", 5, ["B"]),
       ("D", "Method design", 4, ["A"]), ("E", "Prototype implementation", 6, ["C", "D"]), ("F", "Benchmark design", 2, ["D"]),
       ("G", "Experiments", 4, ["E", "F"]), ("H", "Paper writing", 3, ["G"]), ("I", "Workshop preparation", 2, ["G"])]

def cpm(net):
    es, ef = {}, {}
    for t, _, d, p in net:
        es[t] = max([ef[q] for q in p], default=0)
        ef[t] = es[t] + d
    end = max(ef.values())
    ls, lf = {}, {}
    for t, _, d, _ in reversed(net):
        succ = [s for s, _, _, p in net if t in p]
        lf[t] = min([ls[s] for s in succ], default=end)
        ls[t] = lf[t] - d
    table = pd.DataFrame({"ES": es, "EF": ef, "LS": ls, "LF": lf})
    table["float"] = table["LS"] - table["ES"]
    return table, end

schedule, duration = cpm(NET)
schedule

The function assumes that tasks are listed so that every predecessor appears before its successors, as in the list above.

### Your turn, exercise 1

Return the list of critical tasks, those with zero float, in the order of the task list. The project duration is printed below the check.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _critical_reference():
    t, end = cpm(NET)
    return list(t.index[t["float"] == 0])

In [ ]:
critical_tasks = None  # a list of task identifiers
check("Exercise 1", critical_tasks, _critical_reference())
print("project duration:", duration, "weeks")

## 2. PERT probability of meeting a deadline

Assume that the standard deviation of each task is 20 percent of its duration.

### Your turn, exercise 2

Using the critical path A, D, E, G, H only, compute the PERT probability of finishing within 22 weeks with a normal approximation. Explain in two sentences why the second critical path makes this probability too optimistic.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
from scipy.stats import norm
def _pert_prob_reference():
    path = [d for t, _, d, _ in NET if t in "ADEGH"]
    sd = np.sqrt(sum((0.2 * d) ** 2 for d in path))
    return float(norm.cdf((22 - sum(path)) / sd))

In [ ]:
p_deadline = None  # your computation
check("Exercise 2", p_deadline, _pert_prob_reference())

## Visual exploration: The critical path as a Gantt chart

The cell computes early and late start times with a forward and a backward pass, then draws the schedule. Critical tasks are red and total float is hatched. Change a duration and watch the critical path move.

In [ ]:
tasks = {"A": (3, []), "B": (2, []), "C": (4, ["A"]), "D": (5, ["A", "B"]), "E": (2, ["C"]), "F": (3, ["D"]), "G": (2, ["E", "F"])}
es, ef = {}, {}
for t, (d, pre) in tasks.items():
    es[t] = max((ef[p] for p in pre), default=0); ef[t] = es[t] + d
T = max(ef.values())
lf, ls = {}, {}
for t in reversed(list(tasks)):
    succ = [s for s, (_, pre) in tasks.items() if t in pre]
    lf[t] = min((ls[s] for s in succ), default=T); ls[t] = lf[t] - tasks[t][0]
fig, ax = plt.subplots(figsize=(9, 3.8))
for i, t in enumerate(tasks):
    critical = ls[t] == es[t]
    ax.barh(i, tasks[t][0], left=es[t], color="tab:red" if critical else "tab:blue")
    if not critical:
        ax.barh(i, ls[t] - es[t], left=ef[t], color="none", edgecolor="grey", hatch="//")
ax.set_yticks(range(len(tasks)), [f"{t} (float {ls[t] - es[t]})" for t in tasks]); ax.invert_yaxis(); ax.set_xlabel("week")
ax.set_title(f"Critical path in red, total float hatched; duration {T} weeks"); plt.tight_layout(); plt.show()

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/rd-project-management-NB-lecture/weeks/week-09/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Enter your own project in the calculator. Which task on the critical path is most uncertain?
2. Where in your project could fast tracking create more risk than it saves time?
3. Why do administrative tasks so often lie on the critical path of research projects?

## Weekly task and submission

Build the dependency network of your project, compute the critical path and float with the calculator or the notebook, and include the schedule figure in your proposal draft. State the probability of meeting the planned end date under PERT and explain its limitation. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Scheduling beyond the critical path.** Review scheduling approaches that extend the critical path method, such as resource-constrained scheduling and buffer management, and assess their suitability for research projects [1, 2, 3].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Kelley, J. E., & Walker, M. R. (1959). Critical-path planning and scheduling. In *Proceedings of the Eastern Joint Computer Conference* (pp. 160-173). <https://doi.org/10.1145/1460299.1460318>

[2] Malcolm, D. G., Roseboom, J. H., Clark, C. E., & Fazar, W. (1959). Application of a technique for research and development program evaluation. *Operations Research*, 7(5), 646-669. <https://doi.org/10.1287/opre.7.5.646>

[3] Project Management Institute (2021). *A Guide to the Project Management Body of Knowledge (PMBOK Guide) and The Standard for Project Management* (7th ed.). Project Management Institute.